In [ ]:
import os
import numpy as np 
import matplotlib.pyplot as plt
import random
from ase.visualize import view

In [ ]:
# Functions to read and write xyz files

def read_xyz(filename):
    """Read XYZ file and return atom names and coordinates.
    
    Args:
        filename (str): Name of the XYZ file.
        
    Returns:
        atom_names (list): List of element symbols.
        coords (list): List of coordinates for every frame.
    """
    coords = []
    with open(filename, 'r') as f:
        while True:
            try:
                natm = int(f.readline().strip())  # Read number of atoms
                comment = f.readline()  # Skip comment line
                atom_names = []
                frame = []
                for _ in range(natm):
                    parts = f.readline().split()
                    atom_names.append(parts[0])
                    frame.append([float(parts[1]), float(parts[2]), float(parts[3])])
                coords.append(frame)
            except Exception as e:
                # End of file or format error
                break
    return atom_names, coords


def write_xyz(filename, atoms, coords):
    """Write atom names and coordinate data to an XYZ file.
    
    Args:
        filename (str): Name of the output XYZ file.
        atoms (iterable): List of atom names.
        coords (list): Coordinates (list of frames, each frame is a list of atom positions).
    """
    natoms = len(atoms)
    with open(filename, 'w') as f:
        for i, frame in enumerate(coords):
            f.write(f"{natoms}\n")
            f.write(f"Frame {i}\n")
            for a, pos in zip(atoms, frame): 
                f.write(" {:3} {:21.12f} {:21.12f} {:21.12f}\n".format(a, *pos))


In [ ]:
# Define folder paths for each simulation.
# If you have only one dataset, simply define one key. In this example, we have a folder named "ACTIVE_LEARNING_DATA" 
# that contains the "model_devi_summary.out" file and the trajectory .xyz file (specified in the next cell).
folders = {
    "my_unique_dataset": "./path_to_some/ACTIVE_LEARNING_DATA"
}

# Load the model deviation data from each folder into a dictionary.
data_dict = {}
for key, folder in folders.items():
    file_path = os.path.join(folder, "model_devi_summary.out")
    data = np.loadtxt(file_path, skiprows=2,delimiter='|', usecols=1)[:1758]
    data_dict[key] = data
    n_points = len(data)
    sim_time = n_points * 0.00025 * 40
    print(f"{key}: {n_points} data points, simulation time = {sim_time:.2f}")


In [ ]:
# Read trajectories from your .xyz files and store them in dictionaries.
atoms_dict = {}
coords_dict = {}

for key, folder in folders.items():
    xyz_file = os.path.join(folder, "your_trajectory.xyz")  # change this to the name of your .xyz file
    print(f"Reading {xyz_file} ...")
    atoms, coords = read_xyz(xyz_file)
    atoms_dict[key] = atoms
    coords_dict[key] = coords
    print(f"Finished reading {xyz_file}")


In [ ]:
# Compute maximum force deviation (assumed to be in column index 4) for each dataset.
mf_dict = {}
for key, data in data_dict.items():
    mf = data
    mf[mf > 1.25] = 1.25  # Cap maximum force deviation at 1.25 eV/Å
    mf_dict[key] = mf
    print(f"{key}: Average MF = {np.average(mf):.4f} eV/Å")

# Auto-generate colors using a colormap.
n_keys = len(folders)
cmap = plt.get_cmap("tab10")
keys_sorted = sorted(folders.keys())  # Ensure consistent ordering
colors = { key: cmap(i % 10) for i, key in enumerate(keys_sorted) }
labels = { key: key for key in keys_sorted }

# Plot histogram for each dataset.
nbin = 300
plt.figure(figsize=(10, 5))
for key in keys_sorted:
    plt.hist(mf_dict[key], bins=nbin, color=colors[key], alpha=0.3, label=labels[key])
plt.title('Histogram of Maximum Force Deviation')

plt.xlabel('Max Force Deviation (eV/Å)',fontsize='x-large')
plt.ylabel('Frequency',fontsize='x-large')
mstd=np.average(mf_dict[keys_sorted[0]]) - 0.05
plt.axvline(mstd)
plt.axvline(mstd+0.05)
plt.axvline(mstd+0.15)
plt.axvline(mstd+0.25)
plt.legend(fontsize='x-large',loc='upper right')
plt.tight_layout()
plt.xticks(fontsize='large')
plt.yticks(fontsize='large')
plt.show()

In [ ]:
# Choose ONE dataset for interval analysis.
# For instance, if you have only one dataset this automatically picks that one.
hist_key = keys_sorted[0]  # Change index if you want to select a different dataset
mf_selected = mf_dict[hist_key]

# Optionally, compute a characteristic value (e.g., sigma_max) from the histogram.
counts, bin_edges = np.histogram(mf_selected, bins=nbin)
max_index = np.argmax(counts)
sigma_max = bin_edges[max_index]
print(f"{hist_key}: sigma_max = {sigma_max:.4f} eV/Å")

# Define intervals (modify these as needed for your application).
interval_1 = [mstd, mstd+0.05]
interval_2 = [mstd+0.05, mstd+0.15]
interval_3 = [mstd+0.15, mstd+0.25]

# For the dataset selected in cell 5 (hist_key), get indices falling in each interval.
int1_geom = []
int2_geom = []
int3_geom = []

for i, mf_val in enumerate(mf_selected):
    if interval_1[0] <= mf_val < interval_1[1]:
        int1_geom.append(i)
    elif interval_2[0] <= mf_val < interval_2[1]:
        int2_geom.append(i)
    elif interval_3[0] <= mf_val < interval_3[1]:
        int3_geom.append(i)

print(f"Interval {interval_1}: {len(int1_geom)} configurations, {(len(int1_geom)/len(mf_selected))*100:.2f}%")
print(f"Interval {interval_2}: {len(int2_geom)} configurations, {(len(int2_geom)/len(mf_selected))*100:.2f}%")
print(f"Interval {interval_3}: {len(int3_geom)} configurations, {(len(int3_geom)/len(mf_selected))*100:.2f}%")


In [ ]:
# Specify the dataset key from which to extract configurations.
# If you have only one dataset, this will be that key.
extract_key = keys_sorted[0]  # or, for example: "my_unique_dataset"

# Define the number of samples to extract from each interval.
# CHANGE IF YOU WANT A DIFFERENT NUMBER OF SAMPLES.
Nsample_int1 = 30 
Nsample_int2 = 40
Nsample_int3 = 30

# Ensure there are enough configurations in each interval.
if len(int1_geom) < Nsample_int1 or len(int2_geom) < Nsample_int2 or len(int3_geom) < Nsample_int3:
    raise ValueError("Not enough configurations in one of the intervals for the desired sampling.")

# Randomly select frame indices for each interval.
selected_int1 = sorted(random.sample(int1_geom, Nsample_int1))
selected_int2 = sorted(random.sample(int2_geom, Nsample_int2))
selected_int3 = sorted(random.sample(int3_geom, Nsample_int3))

# Define output file paths based on the extraction folder.
folder_extract = folders[extract_key]
output_log = os.path.join(folder_extract, "output_selected.log")
geom_file1 = os.path.join(folder_extract, "geom_int1.xyz")
geom_file2 = os.path.join(folder_extract, "geom_int2.xyz")
geom_file3 = os.path.join(folder_extract, "geom_int3.xyz")
geom_all =  os.path.join(folder_extract, "geom_all.xyz")

# Write the output log using the corresponding model deviation data.
data_extract = data_dict[extract_key]
print(data_extract)
with open(output_log, 'w') as f:
    f.write("# Selected Frame 1 interval:\n#--------------------------\n")
    for idx in selected_int1:
        f.write(f"{idx} {data_extract[idx]:.6f}\n")
    f.write("#--------------------------\n# Selected Frame 2 interval:\n#--------------------------\n")
    for idx in selected_int2:
        f.write(f"{idx} {data_extract[idx]:.6f}\n")
    f.write("#--------------------------\n# Selected Frame 3 interval:\n#--------------------------\n")
    for idx in selected_int3:
        f.write(f"{idx} {data_extract[idx]:.6f}\n")
print("Output log written to:", output_log)

# Extract coordinates and write xyz files for the selected frames.
coords_extract = coords_dict[extract_key]
atoms_extract = atoms_dict[extract_key]

selected_coords1 = [coords_extract[i] for i in selected_int1]
selected_coords2 = [coords_extract[i] for i in selected_int2]
selected_coords3 = [coords_extract[i] for i in selected_int3]

selected_coords_all  = selected_coords1 + selected_coords2 + selected_coords3

print("Writing", geom_file1)
write_xyz(geom_file1, atoms_extract, selected_coords1)
print("Writing", geom_file2)
write_xyz(geom_file2, atoms_extract, selected_coords2)
print("Writing", geom_file3)
write_xyz(geom_file3, atoms_extract, selected_coords3)
print("Writing geom containing all configs")
write_xyz(geom_all, atoms_extract, selected_coords_all)
